# Notebook 02 — SimpleMLP 784→40→20→10 on MNIST digit classification

## Structure
0. **Imports & setup**
1. **Configuration**
2. **Backward Factor Trace** — seed-0 model, BFT, exploratory plots (factor panels, galleries, scaffold, pixel receptive fields)
3. **BFT figures** — main-paper figure 3 and its appendix companion
4. **Fingerprints** — NNLS round-trip, near-OOD (Fashion-MNIST), far-OOD (4 synthetic types), embeddings
5. **Fingerprint figures** — main paper and appendix (placeholders)

Validation, robustness and ablation analyses live in notebook 09.

## §0 — Imports & setup

In [ ]:
#%matplotlib inline
import sys, os
sys.path.insert(0, '..')

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.metrics.pairwise import paired_cosine_distances
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader, TensorDataset

from src import (
    SimpleMLP, train_epoch, evaluate, load_experiment, save_experiment, get_transform,
    get_loaders_from_config, collect_layer_dicts, bft, build_scaffold_edges,
    scaffold_loading_from_edges, scaffold_layer_sizes_from_edges, plot_scaffold_graph,
    extract_tree_nodes, plot_factor_tree, extract_fingerprint_matrix,
    compute_stimulus_similarity, project_stimuli_onto_tree, project_onto_bft,
    extract_factor_tree_nodes, compute_factor_activations, nodes_at_layer,
    plot_factor_overview_panel, plot_factor_gallery, plot_input_layer_factors,
    plot_embedding_comparison,
)
from src.training import train_epoch

plt.rcParams.update({'figure.dpi': 80})
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

## §1 — Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
MODEL_ROOT = '../data/models'
FIG_DIR    = '../figs/02_mlp_40_20_digits'
MODEL_DIR  = os.path.join(MODEL_ROOT, 'mnist_digit_mlp_40_20_seed0')
os.makedirs(MODEL_ROOT, exist_ok=True)
os.makedirs(FIG_DIR,    exist_ok=True)

# ── Model & training config ───────────────────────────────────────────────────
BASE_CONFIG = {
    'arch': 'SimpleMLP',
    'arch_kwargs': {'input_dim': 784, 'hidden_dims': [40, 20], 'output_dim': 10},
    'dataset': 'MNIST',
    'dataset_kwargs': {'root': '../data/', 'batch_size': 64},
    'label_transform': 'identity',
    'analysis_layer_indices': [2, 4, 6],
    'n_per_class': 1000,
    'input_side': 28,
}
N_EPOCHS = 30

# ── BFT hyperparameters — single circuit tree (nb15 C0 held-out selection) ────
K_TRACE         = [10, 11, 14]   # C0 circuit tree (nb15)
B_CIRC          = [1, 2, 14]
STIM_THRESHOLD  = 0.7
# See nb01: the default validate_top_m=100 makes non-root recon a 100-sample estimate.
VALIDATE_TOP_M  = 2000
K_CIRC = K_TRACE
# Single-tree design: the fingerprint is the circuit tree's top-2 slice (§4);
# no separate fingerprint tree is fitted.

N_CLASSES   = 10
CLASS_NAMES = {i: str(i) for i in range(N_CLASSES)}
IMAGE_SIDE  = 28

print('Config ready')
print(f'MODEL_DIR: {MODEL_DIR}')

## §2 — Backward Factor Trace (seed 0)

### 2a — Model, loaders and layer activations

In [ ]:
# ── Loaders and seed-0 model ──────────────────────────────────────────────────
_train_loader, _test_loader = get_loaders_from_config(dict(BASE_CONFIG))
label_transform = get_transform(BASE_CONFIG['label_transform'])  # None for 'identity'

if os.path.exists(os.path.join(MODEL_DIR, 'weights.pt')):
    model0, cfg0 = load_experiment(MODEL_DIR, device=DEVICE)
    print(f'Loaded model from {MODEL_DIR}')
else:
    print('No checkpoint found, training seed 0 …')
    torch.manual_seed(0)
    model0 = SimpleMLP(**BASE_CONFIG['arch_kwargs']).to(DEVICE)
    opt    = torch.optim.Adam(model0.parameters(), lr=3e-3)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=N_EPOCHS)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(N_EPOCHS):
        train_epoch(model0, _train_loader, opt, criterion,
                    label_transform=label_transform, device=DEVICE)
        sched.step()
    cfg0 = dict(BASE_CONFIG,
                description='SimpleMLP 784->40->20->10 on MNIST digit classification, seed 0')
    save_experiment(model0, cfg0, MODEL_DIR)
    print(f'Saved to {MODEL_DIR}')

model0.eval()
_, acc = evaluate(model0, _test_loader, nn.CrossEntropyLoss(),
                  label_transform=label_transform, device=DEVICE)
print(f'Seed-0 test acc = {acc:.4f}')

In [ ]:
# Collect the SAME samples BFT primary mode uses (loader order, only_correct), so
# downstream per-sample arrays stay aligned with tree_root0's img_factors.
_d0 = collect_layer_dicts(model0, _test_loader, label_transform=label_transform,
                          device=DEVICE)

layer_inputs0 = [d['input_fmap'] for d in _d0['layer_data']]
all_targets0  = _d0['targets']
all_images0   = _d0['images']
n_samples0    = len(all_targets0)
layer_names   = {0: 'L0 (784→40)', 1: 'L1 (40→20)', 2: 'Out (20→10)'}
print(f'{n_samples0} samples | layers: {[x.shape for x in layer_inputs0]}')

### 2b — Run BFT

In [ ]:
# ── BFT (seed 0) — the single circuit tree ────────────────────────────────────
# Primary mode: pass (model, loader) so BFT collects internally and validate=True works.
from src import cached_tree
tree_circuits = cached_tree('nb02_circuit', lambda: bft(
    model0, _test_loader, k_max=K_TRACE, n_branches=B_CIRC,
    stimulus_threshold=STIM_THRESHOLD, weighting='img_selectivity',
    validate=True, validate_top_m=VALIDATE_TOP_M, n_jobs=3),
    params=dict(k=K_TRACE, b=B_CIRC, tau=STIM_THRESHOLD, n=n_samples0))
tree_root0 = tree_circuits
print('Validation summary:', tree_root0.validation_summary())


tree_nodes0   = extract_tree_nodes(tree_root0)
factor_nodes0 = extract_factor_tree_nodes(tree_root0)
l0_nodes0     = nodes_at_layer(tree_root0, 0)
print(f'Tree nodes: {len(tree_nodes0)}  factor nodes: {len(factor_nodes0)}')
print(f'Root factors: K={len(tree_root0.root.lambdas)}')

### 2c — Exploratory plots: factor overview panels and galleries

In [ ]:
# ── Plot 1: Factor overview panels ────────────────────────────────────────────
# One figure per factor per BFT tree node: lambda bar, neural heatmap,
# stimulus weight histogram (tab10 per digit class), weighted avg image, top-5 stimuli.
tree_nodes0 = extract_tree_nodes(tree_root0)
os.makedirs(FIG_DIR, exist_ok=True)

for r in tree_nodes0:
    path_label = 'F' + '→F'.join(str(f) for f in r['path']) if r['path'] else 'root'
    figs = plot_factor_overview_panel(r, all_images0, all_targets0, CLASS_NAMES)
    for k, fig in enumerate(figs):
        fname = f"L{r['layer_idx']+1}_{path_label.replace('→','-')}_factor{k}.pdf"
        fig.savefig(os.path.join(FIG_DIR, fname), bbox_inches='tight')
        plt.show(); plt.close(fig)

# ── Plot 4: Per-factor image gallery ──────────────────────────────────────────
# 2 rows: top-10 most active / bottom-10 least active images per factor per node
for r in tree_nodes0:
    path_label = 'F' + '→F'.join(str(f) for f in r['path']) if r['path'] else 'root'
    for k in range(r['img_factors'].shape[1]):
        fig = plot_factor_gallery(r, all_images0, all_targets0, CLASS_NAMES, k=k, n=10)
        fname = f"gallery_L{r['layer_idx']+1}_{path_label.replace('→','-')}_k{k}.pdf"
        fig.savefig(os.path.join(FIG_DIR, fname), bbox_inches='tight')
        plt.show(); plt.close(fig)

### 2d — Exploratory plots: scaffold graph

In [ ]:
# ── Scaffold graph (seed 0) ───────────────────────────────────────────────────
def _get_spine(root):
    chain, node = [root], root
    while node.children:
        node = node.children[0]; chain.append(node)
    return chain

spine = _get_spine(tree_root0.root)
layer_results = list(reversed(spine))          # forward order: input → output
edge_matrices, neg_edge_matrices = build_scaffold_edges(
    layer_results[1:], fi='path', fi_seed=layer_results[0], top_pct=0.05,
)
scaffold_loading = scaffold_loading_from_edges(edge_matrices)
layer_sizes = scaffold_layer_sizes_from_edges(edge_matrices)
fig = plot_scaffold_graph(scaffold_loading, edge_matrices, layer_sizes,
                          neg_edge_matrices=neg_edge_matrices)
fig.axes[0].set_title('Scaffold graph — seed 0 (node colour = dominant class)', fontsize=12)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'scaffold_seed0.pdf'), bbox_inches='tight')
plt.show()


### 2e — Exploratory plots: input-layer pixel receptive fields

In [ ]:
# ── Plot 2: Input-layer pixel receptive fields ────────────────────────────────
# Shows connection_factors[:, k] for each L0 leaf node reshaped to (n_out, 28, 28)
l0_nodes0 = nodes_at_layer(tree_root0, 0)
factor_nodes0 = extract_factor_tree_nodes(tree_root0)

for r in l0_nodes0:
    path_label = 'F' + '→F'.join(str(f) for f in r.path) if r.path else 'root'
    figs = plot_input_layer_factors(r, all_images0, arch='fc',
                                    image_shape=(IMAGE_SIDE, IMAGE_SIDE))
    for k, fig in enumerate(figs):
        fname = f"pixel_rf_L{r.layer_idx+1}_{path_label.replace('→','-')}_k{k}.pdf"
        fig.savefig(os.path.join(FIG_DIR, fname), bbox_inches='tight')
        plt.show(); plt.close(fig)


## §3 — BFT figures (main paper & appendix)

### 3a — Main-paper figure 3 — class circuits

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXPORT — figure data for the 40x20 digit MLP circuits
# Requires: model0, _test_loader, all_targets0, all_images0  (§2 above)
# Writes:   figures/figdata/nb02_circuits.npz  (+ .json)
# The figures are built in notebooks/fig02_mlp_digits.ipynb from this bundle alone.
# ═══════════════════════════════════════════════════════════════════════════════
from src import figdata, figexport

N_DIGITS, IMAGE_SIDE, N_SHOW = 10, 28, 3        # L1 factors shown per circuit

# One single-chain circuit per output-layer factor (reused if already traced).
try:
    tree_circuits
except NameError:
    from src import cached_tree
    tree_circuits = cached_tree('nb02_circuit', lambda: bft(model0, _test_loader,
                        k_max=K_TRACE, n_branches=B_CIRC, stimulus_threshold=STIM_THRESHOLD,
                        weighting='img_selectivity', validate=True, validate_top_m=2000, n_jobs=3),
                        params=dict(k=K_TRACE, b=B_CIRC, tau=STIM_THRESHOLD, n=n_samples0))


def digit_profile(node, k, targets=None):
    """Share of factor k's mean stimulus loading contributed by each digit."""
    targets = all_targets0 if targets is None else targets
    col = node.img_factors[:, k]
    m = np.array([col[targets == d].mean() for d in range(N_DIGITS)])
    return m / (m.sum() + 1e-12)


def digit_profiles(node):
    return np.stack([digit_profile(node, k) for k in range(node.img_factors.shape[1])])


def pixel_arbors(node):
    """(K, 28, 28): pixel-space arbor of every factor of an input-layer node."""
    n_out, n_in = node.weight.shape
    F = node.connection_factors
    return np.stack([F[:, k].reshape(n_out, n_in).sum(0).reshape(IMAGE_SIDE, IMAGE_SIDE)
                     for k in range(F.shape[1])])


def weighted_avg_stimuli(node, flat_images):
    """(K, 28, 28): the stimulus average each factor's loadings define."""
    W = node.img_factors
    return np.stack([(W[:, k, None] * flat_images).sum(0) / (W[:, k].sum() + 1e-12)
                     for k in range(W.shape[1])]).reshape(-1, IMAGE_SIDE, IMAGE_SIDE)


# ── circuits: one per output-layer factor, traced to the input layer ────────
root = tree_circuits.root
LAYER_SIZES = [model0.layers[i].out_features for i in model0.linear_layer_indices()]
_flat = all_images0.reshape(len(all_images0), -1)
CIRCUITS = []
for child in root.children:
    leaf = child.children[0]
    while leaf.children:
        leaf = leaf.children[0]
    ci = int(child.path[0])
    chain = list(reversed([root, child, leaf]))                  # L1-first
    E, negE = build_scaffold_edges(chain[1:], fi='path', fi_seed=chain[0], top_pct=0.05)
    prof = digit_profile(root, ci)
    l1_profiles = digit_profiles(leaf)
    CIRCUITS.append(dict(
        k=ci, profile=prof,
        pooled=[int(d) for d in np.argsort(prof)[::-1] if prof[d] >= 0.15],
        l1_lam=leaf.lambdas / leaf.lambdas.sum(),
        l1_profiles=l1_profiles, l1_pur=l1_profiles.max(1),
        l1_arbors=pixel_arbors(leaf), l1_wavg=weighted_avg_stimuli(leaf, _flat),
        scaffold=dict(edges=E, neg_edges=negE,
                      loading=scaffold_loading_from_edges(E),
                      layer_sizes=scaffold_layer_sizes_from_edges(E)),
        support=np.asarray(scaffold_loading_from_edges(E)[:LAYER_SIZES[0]], float)))
n_c = len(CIRCUITS)

# digit purity at the output layer vs at layer 1, and how much circuits share units
root_pur = np.array([c['profile'].max() for c in CIRCUITS])
Sup = np.stack([c['support'] for c in CIRCUITS])
Sup = Sup / (Sup.sum(1, keepdims=True) + 1e-12)
_Sn = Sup / (np.linalg.norm(Sup, axis=1, keepdims=True) + 1e-12)
_iu = np.triu_indices(n_c, 1)
_rng = np.random.default_rng(0)
_null = [np.mean((lambda P: (P / np.linalg.norm(P, axis=1, keepdims=True)) @
                            (P / np.linalg.norm(P, axis=1, keepdims=True)).T)(
             np.stack([_rng.permutation(r) for r in Sup]))[_iu]) for _ in range(500)]
print(f"digit purity: output {root_pur.mean():.2f} "
      f"(range {root_pur.min():.2f}-{root_pur.max():.2f})  ->  best L1 factor "
      f"{np.mean([c['l1_pur'].max() for c in CIRCUITS]):.2f} "
      f"(range {min(c['l1_pur'].max() for c in CIRCUITS):.2f}-"
      f"{max(c['l1_pur'].max() for c in CIRCUITS):.2f})")
print(f"L1-unit support: {np.mean(1.0 / (Sup ** 2).sum(1)):.0f}/{LAYER_SIZES[0]} "
      f"effective units per circuit, pairwise overlap "
      f"{(_Sn @ _Sn.T)[_iu].mean():.2f} vs {np.mean(_null):.2f} for shuffled units")

# ── superset: the whole trace, a stimulus subsample and a few real images, so
#    panels can be redesigned later without re-running BFT ──────────────────
_stim = figexport.subsample_by_class(all_targets0, range(N_DIGITS), 200, seed=0)

nodes = figexport.export_tree(root, labels=all_targets0, classes=range(N_DIGITS),
                              images=all_images0, stim_idx=_stim,
                              max_matrix=50_000)
D = figdata.save('nb02_circuits', dict(
    n_digits=N_DIGITS, n_show=N_SHOW, layer_sizes=LAYER_SIZES,
    root_lam=root.lambdas / root.lambdas.sum(),
    support=Sup, root_pur=root_pur, circuits=CIRCUITS,
    stim_labels=all_targets0.astype(int),
    images=figexport.stimulus_pool(all_images0, nodes, max_side=28),
    meta=figexport.trace_meta(tree_circuits, k_max=K_TRACE, n_branches=B_CIRC,
                              stimulus_threshold=STIM_THRESHOLD,
                              classes=list(range(N_DIGITS))),
    nodes=nodes,
    stimuli=figexport.example_stimuli(all_images0, all_targets0, range(N_DIGITS),
                                      per_class=8, max_side=28)))
figdata.summary('nb02_circuits')


In [ ]:
# Analysis contract for §6-§9 (identical cells across notebooks).
# ── Single tree: the fingerprint is the circuit tree's top two levels (§4/§5
#    below read tree_root0, so it is rebound to the top-2 slice here).
from src import truncate_tree
tree_root0 = truncate_tree(tree_circuits, depth=2)
print('circuit tree:', sum(1 for _ in tree_circuits.nodes()), 'nodes | '
      'fingerprint (top-2 slice):', sum(1 for _ in tree_root0.nodes()), 'nodes,',
      sum(n.img_factors.shape[1] for n in tree_root0.nodes()), 'dims')

ANALYSIS_CTX = dict(
    exp='mlp_digit', prune_name='nb14_pruning_mlp_digit',
    tag='nb02', model=model0, tree_circuit=tree_circuits, tree_fp=tree_root0,
    layer_inputs=layer_inputs0, labels_task=all_targets0.astype(int),
    labels_fine=all_targets0.astype(int), eval_loader=_test_loader,
    label_transform=label_transform, device=DEVICE,
    layer_names=[d['name'] for d in _d0['layer_data']],
    n_classes=10, k_cap=16, last_extra=4, k_max_cfg=K_CIRC,
    prune_fractions=(0.02, 0.05, 0.1, 0.2), n_random=5, stab_seeds=5)
print('ANALYSIS_CTX ready | circuit', sum(1 for _ in tree_circuits.nodes()),
      'nodes | fp', sum(1 for _ in tree_root0.nodes()), 'nodes')

### 3b — Appendix figure — decomposition details

In [ ]:
# (appendix figure moved to notebooks/fig02_mlp_digits.ipynb)


## §4 — Fingerprints

In [ ]:
# §3 only exports plot data now — the paper figures live in
# notebooks/fig02_mlp_digits.ipynb — so nothing above changed matplotlib's
# rcParams. Kept so the exploratory plots below render at screen size.
plt.rcParams.update({'figure.dpi': 80})


### 4a — NNLS round-trip fidelity

In [ ]:
# ── Round-trip test (NNLS projection fidelity) ────────────────────────────────
N_RT = min(500, n_samples0)
rng_rt = np.random.default_rng(0)
rt_sub = rng_rt.choice(n_samples0, N_RT, replace=False)
rt_inputs = [l[rt_sub] for l in layer_inputs0]

projected_rt = project_stimuli_onto_tree(tree_root0, rt_inputs)
F_orig_rt    = extract_fingerprint_matrix(tree_root0, rt_sub)
F_rt         = extract_fingerprint_matrix(projected_rt, np.arange(N_RT))
rt_sims      = 1.0 - paired_cosine_distances(F_orig_rt, F_rt)
rt_root      = 1.0 - paired_cosine_distances(
    tree_root0.root.img_factors[rt_sub],
    projected_rt.img_factors,
)
print(f'Round-trip (full):   mean={rt_sims.mean():.4f}  std={rt_sims.std():.4f}  min={rt_sims.min():.4f}')
print(f'Round-trip (root):   mean={rt_root.mean():.4f}  std={rt_root.std():.4f}')
print()
print('Active-sample fraction per node (sw > 0.01):')
for tn in tree_nodes0:
    sw = tn.get('stimulus_weights', np.array([1]))
    print(f'  layer={tn["layer_idx"]}  path={tn["path"]}  active={(sw > 0.01).mean():.3f}')


### 4b — Near-OOD: Fashion-MNIST

In [ ]:
# ── Near-OOD: Fashion-MNIST ────────────────────────────────────────────────────
FMNIST_CLASSES = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
                   'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

fmnist_test   = datasets.FashionMNIST('../data/', train=False, download=True,
                                       transform=ToTensor())
fmnist_loader = DataLoader(fmnist_test, batch_size=128, shuffle=False)

projected_ood    = project_onto_bft(tree_root0, model0, fmnist_loader, device=DEVICE)
ood_images       = projected_ood.images
ood_targets      = projected_ood.targets
n_ood            = len(ood_images)
print(f'Near-OOD (Fashion-MNIST) samples: {n_ood}')

factor_nodes_ood = extract_factor_tree_nodes(projected_ood)

# Collect layer activations for MDS comparison (cell 19)
_raw_ood   = collect_layer_dicts(model0, fmnist_loader, DEVICE, only_correct=False)
ood_inputs = [ld['input_fmap'] for ld in _raw_ood['layer_data']]

# What the network itself predicts on Fashion-MNIST — the reference for §5.
with torch.no_grad():
    ood_preds = np.concatenate([
        (lambda o: o[0] if isinstance(o, tuple) else o)(
            model0(x.to(DEVICE))).argmax(1).cpu().numpy()
        for x, _ in fmnist_loader])

# ── Factor tree per Fashion-MNIST class ───────────────────────────────────────
ncols = 5; nrows = 2
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.5 * nrows), squeeze=False)
for i, cl in enumerate(range(10)):
    ax   = axes[i // ncols][i % ncols]
    idx  = np.where(ood_targets == cl)[0]
    acts = compute_factor_activations(factor_nodes_ood, idx)
    plot_factor_tree(factor_nodes_ood, acts, ax=ax,
                     title=f'{FMNIST_CLASSES[cl]}  n={len(idx)}')
plt.suptitle('Near-OOD (Fashion-MNIST) — factor tree per class', fontsize=12, y=1.01)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'near_ood_tree_per_class.pdf'), bbox_inches='tight')
plt.show()

### 4c — Far-OOD: synthetic images

In [ ]:
# ── Far-OOD: synthetic images ──────────────────────────────────────────────────
N_FAR_OOD = 500
rng_f = np.random.default_rng(99)
_chk  = (np.indices((IMAGE_SIDE, IMAGE_SIDE)).sum(0) % 2).astype(np.float32)

far_ood_arrays = {
    'gaussian_noise': np.clip(
        rng_f.normal(0.5, 0.25, (N_FAR_OOD, 1, IMAGE_SIDE, IMAGE_SIDE)).astype(np.float32),
        0, 1),
    'uniform_gray':   np.full((N_FAR_OOD, 1, IMAGE_SIDE, IMAGE_SIDE), 0.5, dtype=np.float32),
    'checkerboard':   np.broadcast_to(
        _chk, (N_FAR_OOD, 1, IMAGE_SIDE, IMAGE_SIDE)).copy().astype(np.float32),
    'inverted_test':  np.clip(1.0 - all_images0[:N_FAR_OOD], 0, 1).astype(np.float32),
}

far_ood_data = {}
for name, imgs in far_ood_arrays.items():
    ds     = TensorDataset(torch.from_numpy(imgs), torch.zeros(len(imgs), dtype=torch.long))
    loader = DataLoader(ds, batch_size=128, shuffle=False)
    raw    = collect_layer_dicts(model0, loader, DEVICE, only_correct=False)
    d = {
        'images':       raw['images'],
        'targets':      raw['targets'],
        'layer_inputs': [ld['input_fmap'] for ld in raw['layer_data']],
    }
    d['projected_root'] = project_onto_bft(tree_root0, model0, loader, only_correct=False, device=DEVICE)
    d['factor_nodes']   = extract_factor_tree_nodes(d['projected_root'])
    far_ood_data[name]  = d
    print(f'{name:20s}  n={len(imgs)}')

# ── Factor tree per far-OOD type ──────────────────────────────────────────────
n_types = len(far_ood_data)
fig, axes = plt.subplots(1, n_types, figsize=(6 * n_types, 4.5), squeeze=False)
for ax, (name, d) in zip(axes[0], far_ood_data.items()):
    acts = compute_factor_activations(d['factor_nodes'], np.arange(len(d['images'])))
    plot_factor_tree(d['factor_nodes'], acts, ax=ax, title=name)
plt.suptitle('Far-OOD — factor tree per synthetic type', y=1.02, fontsize=12)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'far_ood_factor_tree.pdf'), bbox_inches='tight')
plt.show()

### 4d — Fingerprint cross-similarity: ID digits vs Fashion-MNIST

In [ ]:
# ── Cross-similarity: ID digits vs Fashion-MNIST ──────────────────────────────
N_BLOCK = 30; rng_blk = np.random.default_rng(2)
blocks_ood = {}
for cl in range(N_CLASSES):
    idx = rng_blk.choice(np.where(all_targets0 == cl)[0],
                          min(N_BLOCK, (all_targets0 == cl).sum()), replace=False)
    blocks_ood[f'ID-{cl}'] = extract_fingerprint_matrix(tree_root0, idx)
for fc in range(10):
    idx = rng_blk.choice(np.where(ood_targets == fc)[0],
                          min(N_BLOCK, (ood_targets == fc).sum()), replace=False)
    blocks_ood[f'FM-{FMNIST_CLASSES[fc][:5]}'] = extract_fingerprint_matrix(projected_ood, idx)

F_blk = np.concatenate(list(blocks_ood.values()), axis=0)
bl_s  = [len(v) for v in blocks_ood.values()]
bl_st = [0] + list(np.cumsum(bl_s[:-1]))
bl_en = list(np.cumsum(bl_s))
S_blk = compute_stimulus_similarity(F_blk)
ctr   = np.array(bl_st) + np.array(bl_s) / 2
bnd   = np.array(bl_en[:-1])

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(S_blk, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label='Cosine similarity')
for b in bnd:
    ax.axhline(b - 0.5, color='k', lw=1.5); ax.axvline(b - 0.5, color='k', lw=1.5)
ax.set_xticks(ctr); ax.set_xticklabels(list(blocks_ood), rotation=45, ha='right', fontsize=8)
ax.set_yticks(ctr); ax.set_yticklabels(list(blocks_ood), fontsize=8)
ax.set_title('ID digits vs Fashion-MNIST fingerprint cross-similarity')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'near_ood_cross_similarity.pdf'), bbox_inches='tight')
plt.show()

### 4e — Fingerprint embeddings and intra/inter-class similarity

In [ ]:
# ── Plot 7: Embedding comparison — ID / Fashion-MNIST / far-OOD ───────────────
N_EACH = 80
rng_m  = np.random.default_rng(7)

F_parts, act_parts, full_parts, lbl_parts, cond_parts = [], [], [], [], []

id_sub = rng_m.choice(n_samples0, min(N_EACH, n_samples0), replace=False)
F_parts.append(extract_fingerprint_matrix(tree_root0, id_sub))
act_parts.append(layer_inputs0[-1][id_sub])
full_parts.append(np.concatenate([li[id_sub] for li in layer_inputs0], axis=1))
lbl_parts.append(all_targets0[id_sub])
cond_parts.extend(['ID-MNIST'] * len(id_sub))

ood_sub = rng_m.choice(n_ood, min(N_EACH, n_ood), replace=False)
F_parts.append(extract_fingerprint_matrix(projected_ood, ood_sub))
act_parts.append(ood_inputs[-1][ood_sub])
full_parts.append(np.concatenate([li[ood_sub] for li in ood_inputs], axis=1))
lbl_parts.append(ood_targets[ood_sub])
cond_parts.extend(['OOD-FashionMNIST'] * len(ood_sub))

for name, d in far_ood_data.items():
    n  = min(N_EACH, len(d['images']))
    ss = rng_m.choice(len(d['images']), n, replace=False)
    F_parts.append(extract_fingerprint_matrix(d['projected_root'], ss))
    act_parts.append(d['layer_inputs'][-1][ss])
    full_parts.append(np.concatenate([li[ss] for li in d['layer_inputs']], axis=1))
    lbl_parts.append(d['targets'][ss])
    cond_parts.extend([name] * n)

fig = plot_embedding_comparison(
    np.concatenate(F_parts), np.concatenate(act_parts), np.concatenate(lbl_parts),
    CLASS_NAMES, digit_targets=None,
    condition_labels=cond_parts,
    far_ood_conditions=list(far_ood_data.keys()),
    activations_all=np.concatenate(full_parts, axis=0),
    title='ID / Fashion-MNIST / far-OOD fingerprint embeddings',
)
fig.savefig(os.path.join(FIG_DIR, 'embedding_comparison_all_ood.pdf'), bbox_inches='tight')
plt.show(); plt.close(fig)

# ── Fingerprint similarity: intra vs inter class ──────────────────────────────
F_all = extract_fingerprint_matrix(tree_root0, np.arange(n_samples0))
S_all = compute_stimulus_similarity(F_all)
classes = sorted(np.unique(all_targets0))
intra_vals, inter_vals = [], []
for ci, cl in enumerate(classes):
    mask = all_targets0 == cl
    intra = S_all[np.ix_(mask, mask)]
    intra_vals.extend(intra[np.triu_indices_from(intra, k=1)])
    for cl2 in classes[ci + 1:]:
        inter_vals.extend(S_all[np.ix_(mask, all_targets0 == cl2)].ravel())
intra_arr, inter_arr = np.array(intra_vals), np.array(inter_vals)
print(f'Intra-class similarity: {intra_arr.mean():.3f} ± {intra_arr.std():.3f}')
print(f'Inter-class similarity: {inter_arr.mean():.3f} ± {inter_arr.std():.3f}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(intra_arr, bins=60, alpha=0.6, density=True,
        label=f'Intra ({intra_arr.mean():.3f})')
ax.hist(inter_arr, bins=60, alpha=0.6, density=True,
        label=f'Inter ({inter_arr.mean():.3f})')
ax.axvline(intra_arr.mean(), color='C0', ls='--', lw=1.5)
ax.axvline(inter_arr.mean(), color='C1', ls='--', lw=1.5)
ax.set(xlabel='Cosine similarity', ylabel='Density',
       title='Factor fingerprint: intra vs inter-class similarity (test set)')
ax.legend()
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fingerprint_intra_inter.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 7b: Same 4-panel embedding, ID test data only ──────────────────────
F_id   = extract_fingerprint_matrix(tree_root0, np.arange(n_samples0))
act_id = layer_inputs0[-1]
act_id_all = np.concatenate(layer_inputs0, axis=1)

fig_id = plot_embedding_comparison(
    F_id, act_id, all_targets0, CLASS_NAMES,
    activations_all=act_id_all,
    title='ID test data — BFT fingerprint embeddings',
)
fig_id.savefig(os.path.join(FIG_DIR, 'embedding_id_only.pdf'), bbox_inches='tight')
plt.show(); plt.close(fig_id)

## §5 — Fingerprint figures (main paper & appendix)

### 5a — Main-paper figure

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXPORT — figure data for the 40x20 digit MLP fingerprints
# Requires: tree_root0, all_targets0, n_samples0, layer_inputs0  (§2)
#           projected_ood, ood_targets, ood_preds, n_ood  (§4b)
#           far_ood_data  (§4c) and rt_sims (§4a)
# Writes:   figures/figdata/nb02_fingerprints.npz  (+ .json)
# The figures are built in notebooks/fig02_mlp_digits.ipynb from this bundle alone.
# ═══════════════════════════════════════════════════════════════════════════════
from src import figdata, figexport
from src.paper_figures import unit
from sklearn.metrics import silhouette_score
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier

FAR_LABEL = {'gaussian_noise': 'noise', 'uniform_gray': 'gray',
             'checkerboard': 'checker', 'inverted_test': 'inverted'}
FM_SHORT = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
            'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Boot']

# ── fingerprints: in-distribution, Fashion-MNIST, far-OOD ──────────────────
F = extract_fingerprint_matrix(tree_root0, np.arange(n_samples0))
F_ood = extract_fingerprint_matrix(projected_ood, np.arange(n_ood))
F_far = {name: extract_fingerprint_matrix(d['projected_root'],
                                          np.arange(len(d['images'])))
         for name, d in far_ood_data.items()}

# ── map every fingerprint dim to its (layer, root branch, factor), then group
# the columns by circuit. extract_fingerprint_matrix concatenates nodes in BFS
# order, so this reproduces the column order of F.
_dims, _q = [], [tree_root0.root]
while _q:
    _n = _q.pop(0)
    for _k in range(_n.img_factors.shape[1]):
        _dims.append((_n.layer_idx, _n.path[0] if _n.path else -1, _k))
    _q.extend(_n.children)
_dims = np.array(_dims)
_top  = _dims[:, 0].max()
BLOCKS = [[int(r)] + [i for i in range(len(_dims))
                      if _dims[i, 0] != _top and _dims[i, 1] == _dims[r, 2]]
          for r in np.where(_dims[:, 0] == _top)[0]]
COL_ORDER = [c for b in BLOCKS for c in b]
BLK_EDGE  = np.cumsum([len(b) for b in BLOCKS])[:-1]

# ── (c) what the fingerprint says vs what the network says, on Fashion-MNIST
CENT = unit(np.stack([F[all_targets0 == c].mean(0) for c in range(N_CLASSES)]))
near_ood = (unit(F_ood) @ CENT.T).argmax(1)          # nearest ID-digit fingerprint
P_model  = np.stack([np.bincount(ood_preds[ood_targets == c], minlength=N_CLASSES)
                     / (ood_targets == c).sum() for c in range(N_CLASSES)])
P_fprint = np.stack([np.bincount(near_ood[ood_targets == c], minlength=N_CLASSES)
                     / (ood_targets == c).sum() for c in range(N_CLASSES)])
R_OOD = np.corrcoef(P_model.ravel(), P_fprint.ravel())[0, 1]
AGREE = (near_ood == ood_preds).mean()


# ── (d) how tightly each condition's fingerprints cluster ──────────────────
def centroid_cos(Fm):
    """Cosine of every fingerprint to its own condition's mean fingerprint."""
    U = unit(Fm)
    c = U.mean(0)
    return U @ (c / (np.linalg.norm(c) + 1e-12))


COND = ([dict(label='ID test', values=centroid_cos(F), color_key='id_data'),
         dict(label='Fashion-MNIST', values=centroid_cos(F_ood),
              color_key='near_ood')] +
        [dict(label=FAR_LABEL[n], values=centroid_cos(F_far[n]), color_key='far_ood')
         for n in far_ood_data])

SIL = silhouette_score(unit(F), all_targets0, metric='cosine')
print(f'fingerprint dim: {F.shape[1]}   digit silhouette: {SIL:.3f}')
print(f'Fashion-MNIST: r(fingerprint digit, network digit) = {R_OOD:.3f}, '
      f'per-sample agreement {AGREE:.3f} (chance {1 / N_CLASSES:.2f})')
for c in COND:
    print(f"  {c['label']:16s} cos to condition mean: median {np.median(c['values']):.3f}")

# ── (b) the stimulus sample the similarity matrix is drawn from ────────────
PER = 40
rng_v = np.random.default_rng(1)
sel = np.concatenate([rng_v.choice(np.where(all_targets0 == c)[0], PER, replace=False)
                      for c in range(N_CLASSES)])

# ── appendix figure: condition means, digit-likeness, separability, blocks ─
ROWS = ([dict(label=str(c), mean=F[all_targets0 == c].mean(0), color_key='id_data')
         for c in range(N_CLASSES)] +
        [dict(label='Fashion-MNIST', mean=F_ood.mean(0), color_key='near_ood')] +
        [dict(label=FAR_LABEL[n], mean=F_far[n].mean(0), color_key='far_ood')
         for n in far_ood_data])
GROUP = [dict(label='digits', start=0, stop=N_CLASSES),
         dict(label='OOD', start=N_CLASSES, stop=len(ROWS))]

LIKE = [dict(label=lab, values=(unit(X) @ CENT.T).max(1), color_key=ck) for lab, X, ck in
        ([('ID test', F, 'id_data'), ('Fashion-MNIST', F_ood, 'near_ood')] +
         [(FAR_LABEL[n], F_far[n], 'far_ood') for n in far_ood_data])]

# Digit separability against activation controls. Unlike the even/odd net, here
# the penultimate activations are the most class-separable representation — the
# fingerprint's value on this model is interpretability, not separability.
REPS = [('BFT\nfingerprint', unit(F), F.shape[1], 'ours'),
        (r'$L_2$ act.', layer_inputs0[1], layer_inputs0[1].shape[1], '0.55'),
        (r'$L_3$ act.', layer_inputs0[2], layer_inputs0[2].shape[1], '0.55'),
        ('all act.', np.concatenate(layer_inputs0, axis=1),
         sum(li.shape[1] for li in layer_inputs0), '0.55'),
        ('pixels', layer_inputs0[0], layer_inputs0[0].shape[1], '0.55')]
SEP = [dict(label=lab, silhouette=silhouette_score(X, all_targets0, metric='cosine'),
            knn=cross_val_score(KNeighborsClassifier(5, metric='cosine'), X,
                                all_targets0, cv=5).mean(), dim=d, color_key=ck)
       for lab, X, d, ck in REPS]

_fp_sub  = figexport.subsample_by_class(all_targets0, range(N_CLASSES), 200, seed=0)
_ood_sub = figexport.subsample_by_class(ood_targets, range(N_CLASSES), 100, seed=0)

# (c) blocks of ID digits and Fashion-MNIST classes, cross-compared
N_BLOCK = 30
rng_blk = np.random.default_rng(2)
blocks, blabels, bcolors = [], [], []
for c in range(N_CLASSES):
    idx = rng_blk.choice(np.where(all_targets0 == c)[0], N_BLOCK, replace=False)
    blocks.append(F[idx]); blabels.append(str(c)); bcolors.append('id_data')
for c in range(N_CLASSES):
    idx = rng_blk.choice(np.where(ood_targets == c)[0], N_BLOCK, replace=False)
    blocks.append(F_ood[idx]); blabels.append(FM_SHORT[c]); bcolors.append('near_ood')

D = figdata.save('nb02_fingerprints', dict(
    n_classes=N_CLASSES, n_factors=F.shape[1],
    col_order=COL_ORDER, blk_edge=BLK_EDGE, block_sizes=[len(b) for b in BLOCKS],
    fp_mean_by_digit=np.stack([F[all_targets0 == c].mean(0)
                               for c in range(N_CLASSES)]),
    fp_sel=F[sel], sel_per_digit=PER,
    sil=SIL, r_ood=R_OOD, agree=AGREE, p_model=P_model, p_fprint=P_fprint,
    cond=COND, rows=ROWS, group=GROUP, like=LIKE, sep=SEP, rt_sims=rt_sims,
    n_block=N_BLOCK, block_fp=np.concatenate(blocks),
    block_labels=blabels, block_color_keys=bcolors,
    # superset: fingerprint matrices themselves, subsampled (50 factors x 9.6k
    # stimuli would be several MB) plus every OOD condition
    fp=dict(id=F[_fp_sub].astype(np.float32), id_targets=all_targets0[_fp_sub],
            id_index=_fp_sub,
            ood=F_ood[_ood_sub].astype(np.float32), ood_targets=ood_targets[_ood_sub],
            ood_preds=ood_preds[_ood_sub], ood_index=_ood_sub,
            far=[dict(label=FAR_LABEL[n], F=F_far[n][:300].astype(np.float32))
                 for n in far_ood_data])))
figdata.summary('nb02_fingerprints')


In [ ]:
# ── act baseline: the network's own activations on the SAME stimuli, row-aligned
#    with the fingerprint by construction (replaces add_activation_baselines.py).
from src import figdata
from src.separability import pool_activations as _pool
_reps = [{'label': '$L_2$ act.', 'X': _pool(layer_inputs0[1]).astype(np.float32)},
         {'label': '$L_3$ act.', 'X': _pool(layer_inputs0[2]).astype(np.float32)}]
for _r in _reps:
    _r['dim'] = int(_r['X'].shape[1])
_D = figdata.load('nb02_fingerprints')
_D['act'] = {'reps': _reps, 'aligned': 1,
             'index': np.arange(len(all_targets0)), 'labels': np.asarray(all_targets0)}
figdata.save('nb02_fingerprints', _D)
print('act baseline written:', [(r['label'], r['X'].shape) for r in _reps])

### 5b — Appendix figure

In [ ]:
# (appendix figure moved to notebooks/fig02_mlp_digits.ipynb)


## §6 — Hyperparameter check (held-out arbor R²)

Re-derives the per-layer circuit rank with the metric-free C0 rule (`src.hp_selection`), on the circuit tree's own arbors. Confirms the `K_MAX` in §1 sits at the reconstruction plateau; reads no fingerprint metric. Cached.

In [ ]:
from src import node_pos_arbor, nodes_per_layer, select_ranks, cached_result
_C = ANALYSIS_CTX
_npl = nodes_per_layer(_C['tree_circuit'], max_nodes=2)
_arbors = {li: [node_pos_arbor(nd, _C['layer_inputs'][li]) for nd in nds]
           for li, nds in _npl.items()}
hp_sel = cached_result(
    _C['tag'] + '_hpsel',
    lambda: select_ranks(_arbors, _C['labels_task'], k_cap=_C['k_cap'],
                         n_classes=_C['n_classes'], last_extra=_C['last_extra']),
    params=dict(kcap=_C['k_cap'], n=len(_C['labels_task']),
                kmax=list(_C['tree_circuit'].root.lambdas.shape)))
print('held-out K* per layer:', hp_sel['profile']['k_from_criterion'])
print('assembled profile     k_max=%s  n_branches=%s'
      % (hp_sel['profile']['k_max'], hp_sel['profile']['n_branches']))
print('§1 circuit k_max was :', _C.get('k_max_cfg'))

## §7 — Validation (faithfulness + class-relevant structure)

On the **circuit** tree: NMF init-stability per layer, causal-reconstruction fidelity (fc layers only), and the weight-term control (arbor-NMF vs activation-NMF separability). All via `src`; cached.

In [ ]:
# §7 — full validation suite -> logs/results/nb09_<exp>.json (figP_validation).
# Reuses src.validation; the circuit tree was traced with validate=True so causal
# reconstruction is available on fc layers. Cached; also writes the results JSON
# that scripts/build_validation_bundles.py re-encodes into the figure bundle.
import os, json as _json
from src import run_validation, cached_result
_C = ANALYSIS_CTX
val = cached_result(_C['tag'] + '_validation',
    lambda: run_validation(_C['exp'], _C['tree_circuit'], _C['tree_fp'],
                           _C['layer_inputs'], _C['labels_task'], _C['labels_fine'],
                           stab_seeds=_C.get('stab_seeds', 5)),
    params=dict(n=len(_C['labels_task']), exp=_C['exp'], fp='top2'))
_rd = os.path.join(REPO if 'REPO' in dir() else '..', 'logs', 'results')
os.makedirs(_rd, exist_ok=True)
with open(os.path.join(_rd, f"nb09_{_C['exp']}.json"), 'w') as _f:
    _json.dump(val, _f, indent=1)
print('validation written: logs/results/nb09_%s.json' % _C['exp'])
from src import validation_bundle
validation_bundle(_C['exp'], val, source=f"logs/results/nb09_{_C['exp']}.json")
print('figdata bundle written: nb09_%s_validation' % _C['exp'])
_st = val['stability']['per_layer']
print('  NMF stability/layer:', {k: round(v['mean'], 3) for k, v in _st.items()})
if val.get('recon'):
    print('  causal recon preact_R2 (median):', round(val['recon']['overall']['preact_r2']['median'], 3))
_bf = val['separability']['by_fine']
print('  separability by_fine: fp=%.3f  act(matched)=%.3f  (weight-term arbor=%.3f vs act=%.3f)'
      % (_bf['bft_fingerprint']['silhouette'], _bf['act_matched']['silhouette'],
         val['A1_weight_vs_activation']['fingerprint_separability']['arbor_nmf']['silhouette'],
         val['A1_weight_vs_activation']['fingerprint_separability']['activation_nmf']['silhouette']))

## §8 — Causal pruning

Prunes each class circuit's weights in BFT-importance order on the **circuit** tree and measures target vs bystander accuracy (`src.pruning`, wrapping `ablation_sweep`). One seed here; add checkpoints for the full seed×class grid on the cluster. Cached.

In [ ]:
# §8 — causal pruning -> data/results/<prune_name>.json (fig2e / fig6f / figB-d).
# Prunes each class circuit on the CIRCUIT tree; writes the per_obs JSON that
# scripts/build_pruning_bundle.py re-encodes. One seed here; the cluster run adds
# seeds via the loop below (extend _reps with more {seed, model, tree, targets}).
# All aggregation/tests live in src.bundles.pruning_bundle.
import os, json as _json
import numpy as _np
from src import run_pruning, pruning_results_dict, cached_result
_C = ANALYSIS_CTX
if not _C.get('prune_name'):
    print('pruning not wired for this model:', _C.get('skip_pruning') or _C['exp'])
    prune = None
else:
    _reps = [{'seed': 0, 'model': _C['model'], 'tree': _C['tree_circuit'],
              'layer_names': _C.get('layer_names'), 'targets': _C['labels_task']}]
    _targets = list(range(_C['n_classes']))
    _frac = _C.get('prune_fractions', (0.005, 0.01, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5))
    _frac_stat = 0.2
    _ploader = _C.get('prune_eval_loader', _C['eval_loader'])
    prune = cached_result(_C['tag'] + '_pruning',
        lambda: run_pruning(_reps, _ploader, _targets,
                            fractions=_frac,
                            label_transform=_C.get('prune_label_transform', _C.get('label_transform')),
                            pred_transform=_C.get('pred_transform'),
                            device=_C.get('device'), n_random_repeats=_C.get('n_random', 5),
                            frac_stat=_frac_stat, verbose=1),
        params=dict(n=len(_targets), exp=_C['exp'], fractions=list(_frac),
                    seeds=[r['seed'] for r in _reps],
                    n_eval=len(getattr(_ploader, 'dataset', []) or []) or None))
    _rd = os.path.join(REPO if 'REPO' in dir() else '..', 'data', 'results')
    os.makedirs(_rd, exist_ok=True)
    _res = pruning_results_dict(_C['exp'], prune, fractions=_frac, frac_stat=_frac_stat)
    with open(os.path.join(_rd, _C['prune_name'] + '.json'), 'w') as _f:
        _json.dump(_res, _f, indent=1)
    print('pruning written: data/results/%s.json' % _C['prune_name'])
    from src import pruning_bundle
    pruning_bundle(_C['exp'], _res)   # -> figures/figdata (floors reject smoke runs)
    for m in ('bft_top', 'bft_bottom', 'random'):
        _d = [o['baseline'][str(o['target_class'])]
              - o['curves'][m][str(_frac_stat)][str(o['target_class'])]
              for o in prune['per_obs'] if m in o['curves']]
        if _d:
            print('  %-11s target drop@%.1f (mean): %+.3f' % (m, _frac_stat, _np.mean(_d)))

## §9 — Fingerprint separability (C1.8)

On the **fingerprint** tree: is the factor fingerprint more class-separable than the network's own activations, and where in the tree does that live? `src.separability` gives silhouette + kNN for the whole tree, its upper/lower slices, and the penultimate / full-activation baselines (native and dim-matched). Cached.

In [ ]:
from src import separability_evaluate, cached_result
_C = ANALYSIS_CTX
sep = cached_result(
    _C['tag'] + '_separability',
    lambda: separability_evaluate(_C['tree_fp'], _C['labels_fine'], _C['layer_inputs']),
    params=dict(n=len(_C['labels_fine']), tag='fp', fp='top2'))
_n = sep['native']
print('native silhouette:  fp_full=%.3f  output_only=%.3f  top_half=%.3f  spine=%.3f'
      % (_n['fp_full']['sil'], _n.get('fp_output_only', {}).get('sil', float('nan')),
         _n.get('fp_top_half', {}).get('sil', float('nan')), _n.get('fp_spine', {}).get('sil', float('nan'))))
print('activation baselines: penult=%.3f  full=%.3f'
      % (_n['act_penult']['sil'], _n['act_full']['sil']))
_p = sep['paired'].get('fp_full__vs__act_penult')
if _p:
    print('dim-matched @%d: fp(pca)=%.3f vs penult(pca)=%.3f'
          % (_p['match_dim'], _p['A_pca']['sil'], _p['B_pca']['sil']))